In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 05_facts/04_fact_conditions
# Purpose  : fact_conditions - Grain: One row = one condition diagnosis for a patient.
#            Factless/Event Fact (no financial measures here) - acts as the natural Bridge
#            between Patient and Condition to answer "Comorbidity Clustering" queries
#            (following the architectural decision in the Physical Model—no dedicated Bridge
#            table is needed, as this table is naturally Many-to-Many between patient and condition).
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.fact_conditions",
    comment="Fact table (factless/event) - one row per condition diagnosis occurrence",
)
def fact_conditions():
    conditions = dlt.read_stream("covid19_lakehouse.silver.conditions")

    patients = (
        spark.read.table("covid19_lakehouse.gold.dim_patient")
        .filter(F.col("is_current") == True)  # noqa: E712
        .select(F.col("patient_id"), F.col("patient_key"))
    )
    condition_dim = spark.read.table("covid19_lakehouse.gold.dim_condition").select(
        F.col("condition_code"), F.col("condition_key")
    )

    df = (
        conditions
        .join(patients, on="patient_id", how="left")
        .join(condition_dim, on="condition_code", how="left")
    )

    return df.select(
        F.xxhash64("patient_id", "encounter_id", "condition_code", "start_date").alias(
            "condition_fact_key"
        ),
        F.col("patient_key"),
        F.col("encounter_id"),  # Degenerate Dimension
        F.col("condition_key"),
        F.date_format(F.col("start_date"), "yyyyMMdd").cast("int").alias("start_date_key"),
        F.date_format(F.col("stop_date"), "yyyyMMdd").cast("int").alias("stop_date_key"),
        F.when(
            F.col("stop_date").isNotNull(),
            F.datediff(F.col("stop_date"), F.col("start_date")),
        ).alias("condition_duration_days"),
        F.col("is_active"),
    )